## Summary and Next Steps

### What We've Accomplished:

1. **Comprehensive Feature Engineering**: 
   - Text statistics (length, complexity, readability)
   - Sentiment analysis for all text components  
   - Comparative features between responses A and B
   - TF-IDF vectorization for content analysis
   - Bias detection features (verbosity, position)

2. **Neural Network Approach**:
   - Custom PyTorch neural network architecture
   - Multi-layer feedforward network with regularization
   - Comprehensive training with validation monitoring
   - Scaled feature preprocessing for optimal performance

3. **Feature Set**: Over 1,300+ features combining statistical, linguistic, and semantic representations

### Key Insights Expected:
- Length and verbosity differences between responses
- Sentiment and readability impact on preferences  
- Content similarity and semantic differences
- Pattern recognition through deep learning

### Next Steps for Improvement:

1. **Advanced NLP Features**:
   - BERT/RoBERTa embeddings for semantic understanding
   - Transformer-based feature extraction
   - Advanced linguistic analysis

2. **Model Enhancements**:
   - Hyperparameter tuning and architecture search
   - Advanced regularization techniques
   - Cross-validation for robust evaluation

3. **Feature Engineering**:
   - Domain-specific feature engineering
   - Feature selection and dimensionality reduction
   - Advanced bias detection metrics

This approach focuses on leveraging deep learning to capture complex patterns in user preferences while maintaining interpretability through comprehensive feature engineering.

In [ ]:
import pandas as pd
import numpy as np
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import train_test_split
from datasets import Dataset
import torch
import os

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Load data
train_df = pd.read_csv('/kaggle/input/llm-classification-finetuning/train.csv')
test_df = pd.read_csv('/kaggle/input/llm-classification-finetuning/test.csv')

# Combine prompt and responses into a single input text
def format_input(row, include_label=True):
    prompt = row['prompt']
    response_a = row['response_a']
    response_b = row['response_b']
    input_text = f"[PROMPT] {prompt} [RESPONSE_A] {response_a} [RESPONSE_B] {response_b}"
    if include_label:
        if row['winner_model_a'] == 1:
            label = 0
        elif row['winner_model_b'] == 1:
            label = 1
        else:
            label = 2
        return {'text': input_text, 'label': label}
    return {'text': input_text}

# Prepare datasets
train_data = [format_input(row) for _, row in train_df.iterrows()]
test_data = [format_input(row, include_label=False) for _, row in test_df.iterrows()]

# Convert to Hugging Face Dataset
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

# Split train data for validation
train_dataset, val_dataset = train_test_split(train_data, test_size=0.1, random_state=42)
train_dataset = Dataset.from_list(train_dataset)
val_dataset = Dataset.from_list(val_dataset)

# Initialize tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Tokenize datasets
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=512)

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask'])

# Initialize model
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=3
)

# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_lin', 'v_lin'],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)
model = get_peft_model(model, lora_config)

# Define training arguments
training_args = TrainingArguments(
    output_dir='/kaggle/working/results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='/kaggle/working/logs',
    logging_steps=10,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    gradient_accumulation_steps=4,
    fp16=True  # Enable mixed precision training for GPU
)

# Define compute_metrics function for evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean()
    return {'accuracy': accuracy}

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Train the model
trainer.train()

# Generate predictions on test set
predictions = trainer.predict(test_dataset)
pred_probs = torch.softmax(torch.tensor(predictions.predictions), dim=-1).numpy()

# Prepare submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'winner_model_a': pred_probs[:, 0],
    'winner_model_b': pred_probs[:, 1],
    'winner_tie': pred_probs[:, 2]
})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print("Submission file created at /kaggle/working/submission.csv")